# Spam SMS Classification using LSTM (RNN)

This project builds a resume-grade Spam SMS classification system using
LSTM-based sequence modeling with TensorFlow/Keras.

Dataset:
- UCI SMS Spam Collection Dataset

Key challenges addressed:
- Short, noisy text sequences
- Severe class imbalance
- Proper evaluation using Precision, Recall, and F1-score
- Industry-aligned NLP pipeline design


IMPORTS & CONFIGURATION

In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import pickle


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [22]:
# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)


In [23]:
# CONFIG (LOCKED BY DESIGN)
VOCAB_SIZE = 8000
MAX_LEN = 40
EMBED_DIM = 64
LSTM_UNITS = 64
BATCH_SIZE = 32
EPOCHS = 8


LOAD UCI SMS SPAM DATASET

In [38]:
import pandas as pd

df = pd.read_csv(
    "/content/spam.csv",
    encoding="latin-1"
)

# Keep only relevant columns
df = df[['v1', 'v2']]

# Rename columns
df.columns = ['label', 'message']

df.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [30]:
df.head()

,label,message
0,NaN,v2
1,0.0,"Go until jurong point, crazy.. Available only ..."
2,0.0,Ok lar... Joking wif u oni...
3,1.0,Free entry in 2 a wkly comp to win FA Cup fina...
4,0.0,U dun say so early hor... U c already then say...


In [39]:
df.isna().sum()


,0
label,0
message,0


DATA EXPLORATION & IMBALANCE ANALYSIS

In [26]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5573 non-null   object
 1   message  5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [27]:
df['label'].value_counts()


,count
label,
ham,4825
spam,747
v1,1


Observation:
- Dataset is heavily imbalanced
- Accuracy alone will be misleading
- Spam recall must be prioritized


LABEL ENCODING

In [40]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})


In [41]:
df['label'].value_counts()


,count
label,
0,4825
1,747


TRAIN / TEST SPLIT

In [43]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['message'].values,
    df['label'].values,
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

print("Train size:", len(X_train_text))
print("Test size:", len(X_test_text))


Train size: 4457
Test size: 1115


TOKENIZATION (SMS-AWARE, WORD LEVEL)

In [44]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [45]:
VOCAB_SIZE = 8000
MAX_LEN = 40


In [46]:
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>",
    lower=True
)

tokenizer.fit_on_texts(X_train_text)


In [47]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)


PADDING & MASKING

In [48]:
X_train = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print(X_train.shape, X_test.shape)


(4457, 40) (1115, 40)


In [ ]:
HANDLE CLASS IMBALANCE

In [49]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


In [50]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],  # ham
    1: class_weights[1]   # spam
}

class_weight_dict


{0: np.float64(0.577481212749417), 1: np.float64(3.7265886287625416)}

LSTM MODEL (MANY-TO-ONE)

In [51]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


In [52]:
EMBED_DIM = 64
LSTM_UNITS = 64


In [53]:
model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        input_length=MAX_LEN,
        mask_zero=True
    ),
    LSTM(
        LSTM_UNITS,
        dropout=0.2,
        recurrent_dropout=0.2
    ),
    Dense(1, activation="sigmoid")
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [54]:
model.build(input_shape=(None, MAX_LEN))
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │       512,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 545,089 (2.08 MB)

 Trainable params: 545,089 (2.08 MB)

 Non-trainable params: 0 (0.00 B)

COMPILE MODEL

In [55]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]  # accuracy only for monitoring
)


TRAIN MODEL

In [56]:
EPOCHS = 8
BATCH_SIZE = 32


In [57]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict
)


Epoch 1/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 26s 169ms/step - accuracy: 0.6519 - loss: 0.5181 - val_accuracy: 0.9843 - val_loss: 0.1264
Epoch 2/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 37s 168ms/step - accuracy: 0.9853 - loss: 0.0828 - val_accuracy: 0.9877 - val_loss: 0.0641
Epoch 3/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 18s 159ms/step - accuracy: 0.9948 - loss: 0.0414 - val_accuracy: 0.9865 - val_loss: 0.0686
Epoch 4/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 19s 166ms/step - accuracy: 0.9981 - loss: 0.0160 - val_accuracy: 0.9865 - val_loss: 0.0671
Epoch 5/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 18s 160ms/step - accuracy: 0.9987 - loss: 0.0090 - val_accuracy: 0.9843 - val_loss: 0.0716
Epoch 6/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 19s 173ms/step - accuracy: 0.9993 - loss: 0.0067 - val_accuracy: 0.9854 - val_loss: 0.0771
Epoch 7/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 19s 160ms/step - accuracy: 1.0000 - loss: 0.0018 - val_accuracy: 0.9832 - val_loss: 0.0913
Epoch 8/8
112/112 ━━━━━━━━━━━━━━━━━━━━ 21s 161ms/step - accuracy: 1.0000 - loss: 0.0016 - 

PROPER EVALUATION (PRECISION / RECALL / F1)

In [58]:
from sklearn.metrics import classification_report, confusion_matrix


In [59]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)


35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step


In [60]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"]
))


              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       966
        Spam       0.93      0.93      0.93       149

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [61]:
confusion_matrix(y_test, y_pred)


array([[955,  11],
       [ 10, 139]])

OPTIONAL — THRESHOLD TUNING (ADVANCED, OPTIONAL)

In [62]:
# Example: increase spam recall
custom_threshold = 0.35
y_pred_custom = (y_pred_prob >= custom_threshold).astype(int)

print(classification_report(
    y_test,
    y_pred_custom,
    target_names=["Ham", "Spam"]
))


              precision    recall  f1-score   support

         Ham       0.99      0.98      0.98       966
        Spam       0.87      0.93      0.90       149

    accuracy                           0.97      1115
   macro avg       0.93      0.96      0.94      1115
weighted avg       0.97      0.97      0.97      1115



SAVE MODEL & TOKENIZER (FOR DEPLOYMENT)

In [63]:
model.save("spam_lstm_model.h5")


In [64]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


FINAL MANUAL TEST (LABEL OUTPUT ONLY)

In [65]:
def predict_spam_label(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    prob = model.predict(pad, verbose=0)[0][0]
    return "SPAM 🚨" if prob >= 0.5 else "HAM ✅"


In [66]:
predict_spam_label("Congratulations! You have won a free prize. Call now!")


'SPAM 🚨'

In [67]:
predict_spam_label("I'll reach home by 8pm today")


'HAM ✅'